https://github.com/SepehrNoey/Explaining-ResNet50-Predictions-with-LIME-and-SHAP/blob/master/explaining-resnet-50-with-lime.ipynb

In [ ]:
# Run this cell in Google Colab
from IPython.display import Javascript

# JavaScript to click the "connect" button every 60 seconds
js_code = """
function ClickConnect(){
    console.log("Auto-clicking to prevent disconnect...");
    document.querySelector("colab-toolbar-button#connect").click();
}
setInterval(ClickConnect, 60000); // 60000 ms = 60 seconds
"""

display(Javascript(js_code))


In [ ]:
!pip install shap

In [ ]:
!pip install lime
!pip install shap
import json
import os
import numpy as np
import PIL
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torchvision.models import resnet50
import torchvision.transforms as T
import torch.nn.functional as F
import torchvision

# reading lime_image
from lime import lime_image
#import shap

from skimage.segmentation import mark_boundaries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set random seed for reproducibility.
np.random.seed(0)
torch.manual_seed(0)

device ="cuda:0" if torch.cuda.is_available() else "cpu"

In [ ]:
import shap
# loading the model
resnet = resnet50(pretrained=True)

for module in resnet.modules():
    if isinstance(module, torch.nn.ReLU):
        module.inplace = False
resnet = resnet.eval().to(device)

# loading the dataset
X, _ = shap.datasets.imagenet50()

# reading imagenet classes
url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
with open(shap.datasets.cache(url)) as file:
    class_names = [line.strip() for line in file.readlines()]
# print("Number of ImageNet classes:", len(class_names))
# print("Class names:", class_names)

In [ ]:
target_cat_classes = [
    'tabby',
    'tiger cat',
    'Persian cat',
    'Siamese cat',
    'Egyptian cat',
    'leopard',
    'snow leopard',
    'jaguar',
    'lion',
    'tiger',
    'cougar'
]

cat_class_indices = []
print("Finding indices for cat classes:")
for cat_class in target_cat_classes:
    try:
        idx = class_names.index(cat_class)
        cat_class_indices.append((cat_class, idx))
        print(f"  '{cat_class}': Index {idx}")
    except ValueError:
        print(f"  '{cat_class}' not found in ImageNet class names.")

print("\nList of cat class indices:")
print(cat_class_indices)

In [ ]:
import torch
import torchvision.models as models

def load_all_models(device):
    # Słownik na nasze modele
    models_dict = {}

    # 1. ResNet18
    models_dict['resnet18'] = models.resnet18(weights='IMAGENET1K_V1').to(device)

    # 2. MobileNet V3 Large   - zmienić nazwy wszystkich plików z resnet50 na mobile
    #models_dict['mobilenet_v3_large'] = models.mobilenet_v3_large(weights='IMAGENET1K_V1').to(device)

    # 3. EfficientNet B0
    #models_dict['efficientnet_b0'] = models.efficientnet_b0(weights='IMAGENET1K_V1').to(device)

    # 4. EfficientNet B3
    #models_dict['efficientnet_b3'] = models.efficientnet_b3(weights='IMAGENET1K_V1').to(device)

    # 5. SqueezeNet 1.1
    #models_dict['squeezenet'] = models.squeezenet1_1(weights='IMAGENET1K_V1').to(device)

    # 6. ResNet50
    #models_dict['resnet50'] = models.resnet50(weights='IMAGENET1K_V1').to(device)

    # 7. VGG16
    #models_dict['vgg16'] = models.vgg16(weights='IMAGENET1K_V1').to(device)

    # Przełączamy wszystkie modele w tryb ewaluacji (wyłączamy naukę)
    for name in models_dict:
        models_dict[name].eval()

    return models_dict

# Inicjalizacja
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
my_models = load_all_models(device)
print(f"Załadowano {len(my_models)} modeli  ")

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/XAI_Project/results/combined_predictions/all_models_combined_cat_predictions_without_nan.csv'
df_probabilities = pd.read_csv(file_path)

#print("Probabilities loaded into df_probabilities DataFrame. First 5 rows:")
#print(df_probabilities.head())

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

# Define the base path for images
IMAGE_BASE_PATH = '/content/drive/MyDrive/XAI_Project/cats_images'

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from torchvision import models
import numpy as np
import cv2
import requests
from PIL import Image
import time


In [ ]:
# pip install lime torch torchvision pillow scikit-image numba

import torch
import torchvision.models as models
import torchvision.transforms as T
import numpy as np
import random
from PIL import Image
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt
from numba import njit
import requests
from io import BytesIO

# -----------------------------
# 0. Ustawienie ziarna losowości
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# -----------------------------
# 1. Przyspieszone generowanie masek Numba (deterministyczne)
# -----------------------------
@njit(fastmath=True, cache=True)
def generate_masks(num_samples, num_superpixels, seed):
    np.random.seed(seed)  # ustawienie ziarna w Numba
    return np.random.randint(0, 2, size=(num_samples, num_superpixels), dtype=np.uint8)

# -----------------------------
# 2. Model PyTorch (GPU jeśli dostępne)
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.eval().to(device)

# Transformacja obrazu
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

# -----------------------------
# 3. Funkcja predykcji w batchach
# -----------------------------
def predict_fn(images_np):
    imgs_torch = torch.stack([
        transform(Image.fromarray(img.astype(np.uint8))) for img in images_np
    ]).to(device)

    with torch.no_grad():
        outputs = model(imgs_torch)
        probs = torch.nn.functional.softmax(outputs, dim=1)
    return probs.cpu().numpy()

# -----------------------------
# 4. Pobranie przykładowego obrazu
# -----------------------------
img_path = "/content/1773665077339 (1).jpg"
img = Image.open(img_path).convert("RGB").resize((224, 224))
img_np = np.array(img)

# -----------------------------
# 5. LIME z przyspieszeniem i powtarzalnością
# -----------------------------
explainer = lime_image.LimeImageExplainer(random_state=SEED)  # ustawienie ziarna w LIME

explanation = explainer.explain_instance(
    image=img_np.astype('double'),
    classifier_fn=predict_fn,
    top_labels=1,
    hide_color=0,
    num_samples=2000
)

# -----------------------------
# 6. Wizualizacja
# -----------------------------
top_label = explanation.top_labels[0]
temp, mask = explanation.get_image_and_mask(
    label=top_label,
    positive_only=True,
    hide_rest=True,
    num_features=5,
    min_weight=0.0
)

plt.figure(figsize=(8, 8))
plt.imshow(mark_boundaries(temp / 255.0, mask))
#plt.title(f"LIME (PyTorch + Numba, SEED={SEED}) dla label {top_label}")
plt.axis('off')
plt.show()


In [ ]:
# resize and take the center part of image to what our model expects
def pil_to_torch(img):
    transf = T.Compose([
        T.Resize((256, 256)),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])
    # unsqeeze converts single image to batch of 1
    return transf(img).unsqueeze(0)

def pil_transform(img):
    transf = T.Compose([
        T.Resize((256, 256)),
        T.CenterCrop(224)
    ])

    return transf(img)

In [ ]:
def cnn_predict(images):
    transf = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
        ])

    batch = torch.stack(tuple(transf(img) for img in images), dim=0)

    logits = resnet(batch.to(device)).cpu()
    probs = F.softmax(logits, dim=1)
    return probs.detach().cpu().numpy()

In [ ]:
file_to_find = '/content/1773407235573.jpeg'
# Find the row(s) where the 'filename' column matches the file_to_find
found_row = df_probabilities[df_probabilities['filename'] == file_to_find]

if not found_row.empty:
    # Get the index of the first matching row
    index_of_file = found_row.index[0]
    print(f"The index for file '{file_to_find}' is: {index_of_file}")
else:
    print(f"File '{file_to_find}' not found in the DataFrame.")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
import numpy as np
import torch
import torchvision.transforms as T
import torch.nn.functional as F
from lime import lime_image
from skimage.segmentation import mark_boundaries

# Assuming my_models, device, class_names, IMAGE_BASE_PATH, pil_transform, pil_to_torch are already defined from previous cells.

def find_image_path_recursive(base_path, filename):
    for root, _, files in os.walk(base_path):
        if filename in files:
            return os.path.join(root, filename)
    return None

current_model_name = 'resnet18'

# Re-define lime_predict_fn to use the efficientnet_b0 model
def lime_predict_fn_for_efficientnet(images_np):
    # LIME explainer passes images as numpy arrays, scaled to [0, 1]
    # We need to convert them to PIL Image, then apply transformations and normalize
    images_pil = [Image.fromarray((img * 255).astype(np.uint8)) for img in images_np]

    transf = T.Compose([
        T.Resize((256, 256)),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])

    batch = torch.stack(tuple(transf(img) for img in images_pil), dim=0)

    # Use the specified model from the loaded models dictionary
    model = my_models[current_model_name]
    logits = model(batch.to(device)).cpu()
    probs = F.softmax(logits, dim=1)
    return probs.detach().cpu().numpy()

# Define output directories for the single LIME explanation
LIME_OUTPUT_BASE_DIR_SINGLE = f'/content/drive/MyDrive/XAI_Project/results_single_explanation/{current_model_name}/LIME'
VIS_DIR_SINGLE = os.path.join(LIME_OUTPUT_BASE_DIR_SINGLE, 'Visualizations')
RAW_DATA_DIR_SINGLE = os.path.join(LIME_OUTPUT_BASE_DIR_SINGLE, 'Raw_Masks')

os.makedirs(VIS_DIR_SINGLE, exist_ok=True);
os.makedirs(RAW_DATA_DIR_SINGLE, exist_ok=True);

# LIME explainer instance
explainer = lime_image.LimeImageExplainer()

file_to_explain = '/content/1773407235573.jpeg'
image_path = '/content/1773407235573.jpeg'

if image_path is None:
    print(f"Error: Image file '{file_to_explain}' not found in '{IMAGE_BASE_PATH}' or its subdirectories.")
elif current_model_name not in my_models:
    print(f"Error: Model '{current_model_name}' not found in 'my_models'. Please ensure it's loaded.")
else:
    try:
        img_pil = Image.open(image_path).convert('RGB')

        # Prepare image for LIME (numpy array in [0,1] range)
        processed_img_pil = pil_transform(img_pil)
        img_array_for_lime = np.array(processed_img_pil)

        # Get the actual prediction from the efficientnet_b0 model for this image
        input_tensor_for_prediction = pil_to_torch(img_pil).to(device)
        with torch.no_grad():
            outputs = my_models[current_model_name](input_tensor_for_prediction)
            probabilities = F.softmax(outputs, dim=1)
            top_prob, top_idx = torch.topk(probabilities, 1)

        predicted_class_index = top_idx.item()
        predicted_class_name = class_names[predicted_class_index]

        print(f"\nModel '{current_model_name}' predicted for {file_to_explain}: {predicted_class_name} (Index: {predicted_class_index}, Probability: {top_prob.item():.4f})")

        # Generate LIME explanation for the predicted class
        print(f"Generating LIME explanation for {file_to_explain} for predicted class '{predicted_class_name}'...")
        explanation = explainer.explain_instance(
            img_array_for_lime,
            lime_predict_fn_for_efficientnet, # Use the corrected predict function
            labels=[predicted_class_index],   # Explain for the model's top predicted class
            hide_color=0,
            num_samples=250
        )

        # Get image and mask for the visualization
        temp_without_bg, mask_without_bg = explanation.get_image_and_mask(
            predicted_class_index,
            positive_only=True,
            negative_only=False,
            num_features=5,
            hide_rest=True
        )

        # Create visualization (mark_boundaries returns float [0,1])
        img_boundry_without_bg = mark_boundaries(temp_without_bg / 255.0, mask_without_bg)

        # Define base name for saving
        base_name = f"{os.path.splitext(file_to_explain)[0]}_{current_model_name}_{predicted_class_name.replace(' ', '_')}"

        # Save visualization
        output_vis_path = os.path.join(VIS_DIR_SINGLE, f"{base_name}_lime_vis.jpg")
        plt.imsave(output_vis_path, img_boundry_without_bg)
        print(f"Saved LIME explanation visualization to {output_vis_path}")

        # Save raw mask data
        output_raw_path = os.path.join(RAW_DATA_DIR_SINGLE, f"{base_name}_lime_raw.npy")
        np.save(output_raw_path, mask_without_bg)
        print(f"Saved LIME explanation raw mask to {output_raw_path}")

    except Exception as e:
        print(f"Error processing {file_to_explain}: {e}")

In [ ]:
display()

In [ ]:
# Define output directories for LIME explanations
LIME_OUTPUT_BASE_DIR = '/content/drive/MyDrive/XAI_Project/results_multi_cam/mobilenet_v3_large/LIME'
VIS_DIR = os.path.join(LIME_OUTPUT_BASE_DIR, 'Visualizations')
RAW_DATA_DIR = os.path.join(LIME_OUTPUT_BASE_DIR, 'Raw_Masks')

os.makedirs(VIS_DIR, exist_ok=True);
os.makedirs(RAW_DATA_DIR, exist_ok=True);

# LIME explainer instance
explainer = lime_image.LimeImageExplainer()

# Prediction function adapted for LIME (expects numpy array [0,1], outputs probabilities)
def lime_predict_fn(images_np):
    # LIME explainer passes images as numpy arrays, scaled to [0, 1]
    # We need to convert them to PIL Image, then apply transformations and normalize
    images_pil = [Image.fromarray((img * 255).astype(np.uint8)) for img in images_np]

    transf = T.Compose([
        T.Resize((256, 256)),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])

    batch = torch.stack(tuple(transf(img) for img in images_pil), dim=0)

    # Use the squeezenet model from the loaded models
    model = my_models['mobilenet_v3_large']
    logits = model(batch.to(device)).cpu()
    probs = F.softmax(logits, dim=1);
    return probs.detach().cpu().numpy()

def find_image_path_recursive(base_path, filename):
    for root, _, files in os.walk(base_path):
        if filename in files:
            return os.path.join(root, filename)
    return None

import time

num_images_to_process = len(df_probabilities.iloc[0:599])
print(f"Starting LIME explanation generation for {num_images_to_process} images...")

start_total_time = time.time()

for i, (index, row) in enumerate(df_probabilities.iloc[0:599].iterrows()): # Changed to process images from index 10 to 20
    filename = row['filename']
    image_path = find_image_path_recursive(IMAGE_BASE_PATH, filename)

    if image_path is None:
        print(f"Error: Image file '{filename}' not found in '{IMAGE_BASE_PATH}' or its subdirectories. Skipping.")
        continue

    try:
        img_pil = Image.open(image_path).convert('RGB')

        # Apply pil_transform to the image first, as LIME expects images to be of consistent size
        # and it internally converts to numpy array (H, W, C)
        processed_img_pil = pil_transform(img_pil)
        img_array_for_lime = np.array(processed_img_pil) # LIME takes numpy array as input

        # Get the predicted class name from df_probabilities FIRST
        predicted_class_name_from_df = row['resnet18_top_1_class']

        target_class_index = None # Initialize

        # First, try to find the index in cat_class_indices
        for cat_name, cat_idx in cat_class_indices:
            if cat_name == predicted_class_name_from_df:
                target_class_index = cat_idx
                break

        # If not found in cat_class_indices, fall back to the full ImageNet class_names
        if target_class_index is None:
            try:
                target_class_index = class_names.index(predicted_class_name_from_df)
            except ValueError:
                print(f"Warning: Predicted class '{predicted_class_name_from_df}' not found in ImageNet class names. Skipping {filename}.")
                continue

        # Generate LIME explanation, explicitly requesting explanation for target_class_index
        explanation = explainer.explain_instance(
            img_array_for_lime,
            lime_predict_fn,
            labels=[target_class_index], # Pass the specific label to explain
            hide_color=0,
            num_samples=250
        )

        # Use this target_class_index for generating the explanation mask
        temp_without_bg, mask_without_bg = explanation.get_image_and_mask(
            target_class_index, # Use the class index from df_probabilities
            positive_only=True,
            negative_only=False,
            num_features=5,
            hide_rest=True
        )

        # Create visualization (mark_boundaries returns float [0,1])
        img_boundry_without_bg = mark_boundaries(temp_without_bg / 255.0, mask_without_bg)

        # Define base name for saving
        base_name = f"{os.path.splitext(filename)[0]}_mobilenet_v3_large_{predicted_class_name_from_df.replace(' ', '_')}"

        # Save visualization
        plt.imsave(os.path.join(VIS_DIR, f"{base_name}_lime_vis.jpg"), img_boundry_without_bg)

        # Save raw mask data
        np.save(os.path.join(RAW_DATA_DIR, f"{base_name}_lime_raw.npy"), mask_without_bg)

        print(f"Processed {filename}: Saved LIME explanation for class '{predicted_class_name_from_df}'")

    except Exception as e:
        print(f"Error processing {filename}: {e}")

end_total_time = time.time()
total_elapsed_time = end_total_time - start_total_time

#print("Finished generating LIME explanations.")
print(f"Total time taken for {num_images_to_process} images: {total_elapsed_time:.2f} seconds")
#if num_images_to_process > 0:
#    avg_time_per_image = total_elapsed_time / num_images_to_process:
#    print(f"Average time per image: {avg_time_per_image:.2f} seconds")

In [ ]:
explanation.top_labels[0]

In [ ]:
print(class_names[351])